In [ ]:
# ============================================================
# POSITIVE INTERACTION EVENT_TYPE USER STATS
#
# Goal:
# - Count số event theo từng positive event_type
# - Count số unique user dùng từng positive event_type
# - Vì 1 user có thể dùng nhiều event_type:
#   + thống kê combo event_type theo user
#   + thống kê overlap matrix giữa các event_type
#
# Output:
# /kaggle/working/positive_interaction_stats/
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import matplotlib.pyplot as plt

# ============================================================
# 0. CONFIG
# ============================================================

EVENTS_PATH = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2/fact_user_events"

OUTPUT_DIR = "/kaggle/working/positive_interaction_stats"
PART_DIR = os.path.join(OUTPUT_DIR, "user_event_type_parts")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PART_DIR, exist_ok=True)

FORCE_REBUILD = True

BATCH_SIZE_EVENTS = 500_000   # nếu crash RAM thì giảm 200_000 hoặc 300_000

# Official positive interaction có cả other_interaction.
# Nhưng nếu bạn chỉ muốn contact channel thật sự thì để False.
INCLUDE_OTHER_INTERACTION = False


POSITIVE_EVENT_TYPES = [
    "view_phone",
    "contact_chat",
    "contact_zalo",
    "contact_sms",
]

if INCLUDE_OTHER_INTERACTION:
    POSITIVE_EVENT_TYPES.append("other_interaction")

POSITIVE_EVENT_TYPES = list(dict.fromkeys(POSITIVE_EVENT_TYPES))

print("POSITIVE_EVENT_TYPES:", POSITIVE_EVENT_TYPES)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

# ============================================================
# 1. HELPERS
# ============================================================

def save_csv(df, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {path} | shape={df.shape}")
    return path


def display_top(df, n=20, title=None):
    if title:
        print("\n" + "=" * 120)
        print(title)
        print("=" * 120)

    if df is None or len(df) == 0:
        print("Không có dữ liệu.")
        return

    try:
        display(df.head(n))
    except Exception:
        print(df.head(n))


def clean_id_series(s, unknown="UNKNOWN"):
    return s.where(s.notna(), unknown).astype(str).str.strip().replace("", unknown)


def normalize_event_type(s):
    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .replace("", "unknown")
    )


# ============================================================
# 2. CLEAN OLD CACHE
# ============================================================

if FORCE_REBUILD:
    for f in os.listdir(PART_DIR):
        if f.endswith(".parquet"):
            os.remove(os.path.join(PART_DIR, f))

# ============================================================
# 3. SCAN fact_user_events MEMORY-SAFE
# ============================================================

if FORCE_REBUILD or len([f for f in os.listdir(PART_DIR) if f.endswith(".parquet")]) == 0:
    print("[BUILD] scanning fact_user_events...")

    event_ds = ds.dataset(EVENTS_PATH, format="parquet")
    available_cols = set(event_ds.schema.names)

    wanted_cols = [
        "user_id",
        "item_id",
        "event_type",
        "date",
        "event_ts",
        "is_contact",
        "device",
        "surface",
    ]

    cols = [c for c in wanted_cols if c in available_cols]

    if "user_id" not in cols:
        raise ValueError("fact_user_events thiếu user_id.")
    if "event_type" not in cols:
        raise ValueError("fact_user_events thiếu event_type.")

    dq = {
        "total_rows": 0,
        "missing_user_id_rows": 0,
        "missing_event_type_rows": 0,
        "positive_event_rows": 0,
        "partial_files": 0,
    }

    part_id = 0

    for i, batch in enumerate(
        event_ds.to_batches(columns=cols, batch_size=BATCH_SIZE_EVENTS),
        start=1
    ):
        df = batch.to_pandas()

        for c in wanted_cols:
            if c not in df.columns:
                df[c] = np.nan

        dq["total_rows"] += len(df)
        dq["missing_user_id_rows"] += int(df["user_id"].isna().sum())
        dq["missing_event_type_rows"] += int(df["event_type"].isna().sum())

        df = df[
            df["user_id"].notna()
            & df["event_type"].notna()
        ].copy()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        df["user_id"] = clean_id_series(df["user_id"], "UNKNOWN_USER")
        df["item_id"] = clean_id_series(df["item_id"], "UNKNOWN_ITEM")
        df["event_type"] = normalize_event_type(df["event_type"])

        # chỉ giữ positive event types
        df = df[df["event_type"].isin(POSITIVE_EVENT_TYPES)].copy()

        dq["positive_event_rows"] += len(df)

        if len(df) == 0:
            del df
            gc.collect()
            continue

        # Date optional
        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
        else:
            df["date"] = pd.NaT

        # user-event_type aggregate trong từng batch
        # giữ event_count, unique item, active days
        part = (
            df.groupby(["user_id", "event_type"], sort=False, dropna=False)
            .agg(
                event_count=("event_type", "size"),
                unique_items=("item_id", "nunique"),
                active_days=("date", "nunique"),
            )
            .reset_index()
        )

        part_id += 1
        part_path = os.path.join(PART_DIR, f"user_event_type_part_{part_id:05d}.parquet")
        part.to_parquet(part_path, index=False)

        dq["partial_files"] = part_id

        if i % 10 == 0:
            print(
                "batches:", i,
                "| raw rows:", dq["total_rows"],
                "| positive rows:", dq["positive_event_rows"],
                "| parts:", part_id
            )

        del df, part
        gc.collect()

    dq_df = pd.DataFrame([dq])
    display_top(dq_df, title="Scan DQ")
    save_csv(dq_df, "00_scan_dq.csv")

else:
    print("[SKIP] partial files exist.")


# ============================================================
# 4. FINAL AGGREGATE WITH DUCKDB
# ============================================================

try:
    import duckdb
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

part_files = [
    os.path.join(PART_DIR, f)
    for f in os.listdir(PART_DIR)
    if f.endswith(".parquet")
]

if len(part_files) == 0:
    raise ValueError("Không có positive interaction nào được tìm thấy.")

duckdb_path = os.path.join(OUTPUT_DIR, "positive_interaction_stats.duckdb")
if FORCE_REBUILD and os.path.exists(duckdb_path):
    os.remove(duckdb_path)

con = duckdb.connect(duckdb_path)
con.execute("PRAGMA threads=4")
con.execute("PRAGMA memory_limit='3GB'")

part_glob = os.path.join(PART_DIR, "*.parquet").replace("\\", "/")

# Final user-event_type table
user_event_type_path = os.path.join(OUTPUT_DIR, "user_event_type_summary.parquet").replace("\\", "/")

con.execute(f"""
    COPY (
        SELECT
            user_id,
            event_type,
            SUM(event_count) AS event_count,
            SUM(unique_items) AS unique_items_sum_by_batch,
            SUM(active_days) AS active_days_sum_by_batch
        FROM read_parquet('{part_glob}')
        GROUP BY user_id, event_type
    )
    TO '{user_event_type_path}'
    (FORMAT PARQUET)
""")

# Event type summary
event_type_summary = con.execute(f"""
    WITH uet AS (
        SELECT *
        FROM read_parquet('{user_event_type_path}')
    ),
    total_users AS (
        SELECT COUNT(DISTINCT user_id) AS total_positive_users
        FROM uet
    )
    SELECT
        event_type,
        COUNT(DISTINCT user_id) AS unique_users,
        SUM(event_count) AS total_events,
        AVG(event_count) AS avg_events_per_user,
        MEDIAN(event_count) AS median_events_per_user,
        MAX(event_count) AS max_events_by_one_user,
        COUNT(DISTINCT user_id) * 1.0 / NULLIF((SELECT total_positive_users FROM total_users), 0) AS user_share_among_positive_users
    FROM uet
    GROUP BY event_type
    ORDER BY unique_users DESC
""").df()

save_csv(event_type_summary, "01_event_type_user_summary.csv")
display_top(event_type_summary, title="Positive event_type user summary")

# Overall user stats
overall_user_stats = con.execute(f"""
    WITH uet AS (
        SELECT *
        FROM read_parquet('{user_event_type_path}')
    ),
    user_level AS (
        SELECT
            user_id,
            COUNT(DISTINCT event_type) AS n_positive_event_types,
            SUM(event_count) AS total_positive_events
        FROM uet
        GROUP BY user_id
    )
    SELECT
        COUNT(*) AS positive_users,
        SUM(total_positive_events) AS positive_events,
        AVG(n_positive_event_types) AS avg_event_types_per_user,
        MEDIAN(n_positive_event_types) AS median_event_types_per_user,
        AVG(total_positive_events) AS avg_positive_events_per_user,
        MEDIAN(total_positive_events) AS median_positive_events_per_user,
        SUM(CASE WHEN n_positive_event_types >= 2 THEN 1 ELSE 0 END) AS users_using_2plus_event_types,
        SUM(CASE WHEN n_positive_event_types >= 2 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS users_using_2plus_event_types_rate
    FROM user_level
""").df()

save_csv(overall_user_stats, "02_overall_positive_user_stats.csv")
display_top(overall_user_stats, title="Overall positive user stats")

# User combo: 1 user có thể dùng nhiều event_type
combo_summary = con.execute(f"""
    WITH uet AS (
        SELECT *
        FROM read_parquet('{user_event_type_path}')
    ),
    user_combo AS (
        SELECT
            user_id,
            STRING_AGG(event_type, ' + ' ORDER BY event_type) AS event_type_combo,
            COUNT(DISTINCT event_type) AS n_event_types,
            SUM(event_count) AS total_events
        FROM uet
        GROUP BY user_id
    )
    SELECT
        event_type_combo,
        n_event_types,
        COUNT(*) AS users,
        SUM(total_events) AS total_events,
        COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () AS user_share
    FROM user_combo
    GROUP BY event_type_combo, n_event_types
    ORDER BY users DESC
""").df()

save_csv(combo_summary, "03_user_event_type_combo_summary.csv")
display_top(combo_summary, 30, "Top user event_type combos")

# Distribution: user dùng bao nhiêu loại positive event_type
n_type_distribution = con.execute(f"""
    WITH uet AS (
        SELECT *
        FROM read_parquet('{user_event_type_path}')
    ),
    user_level AS (
        SELECT
            user_id,
            COUNT(DISTINCT event_type) AS n_positive_event_types
        FROM uet
        GROUP BY user_id
    )
    SELECT
        n_positive_event_types,
        COUNT(*) AS users,
        COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () AS user_share
    FROM user_level
    GROUP BY n_positive_event_types
    ORDER BY n_positive_event_types
""").df()

save_csv(n_type_distribution, "04_n_event_type_distribution.csv")
display_top(n_type_distribution, title="How many positive event types per user")

# ============================================================
# 5. OVERLAP MATRIX
# ============================================================

# Load final user_event_type summary vừa đủ vì đã aggregate rồi
uet = pd.read_parquet(user_event_type_path)

# tạo user x event_type matrix, value = 1 nếu user từng dùng event_type đó
user_type_flag = (
    uet.assign(flag=1)
    .pivot_table(
        index="user_id",
        columns="event_type",
        values="flag",
        aggfunc="max",
        fill_value=0
    )
)

# bảo đảm đủ cột theo đúng thứ tự config
for et in POSITIVE_EVENT_TYPES:
    if et not in user_type_flag.columns:
        user_type_flag[et] = 0

user_type_flag = user_type_flag[POSITIVE_EVENT_TYPES].astype(int)

# overlap count: cell[i, j] = số user dùng cả i và j
overlap_matrix = pd.DataFrame(
    user_type_flag.T.values @ user_type_flag.values,
    index=POSITIVE_EVENT_TYPES,
    columns=POSITIVE_EVENT_TYPES
)

overlap_matrix.to_csv(
    os.path.join(OUTPUT_DIR, "05_event_type_overlap_matrix.csv"),
    encoding="utf-8-sig"
)

display_top(overlap_matrix.reset_index().rename(columns={"index": "event_type"}), title="Event type overlap matrix")

con.close()

# ============================================================
# 6. PLOTS
# ============================================================

# 6.1 Bar chart: unique users per event_type
plot_df = event_type_summary.sort_values("unique_users", ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(plot_df["event_type"], plot_df["unique_users"])
plt.title("Unique users by positive event_type")
plt.xlabel("Unique users")
plt.ylabel("Event type")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_01_unique_users_by_event_type.png"), dpi=160)
plt.show()

# 6.2 Bar chart: total events per event_type
plot_df = event_type_summary.sort_values("total_events", ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(plot_df["event_type"], plot_df["total_events"])
plt.title("Total events by positive event_type")
plt.xlabel("Total events")
plt.ylabel("Event type")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_02_total_events_by_event_type.png"), dpi=160)
plt.show()

# 6.3 Combo chart: top combos
top_combo = combo_summary.head(15).sort_values("users", ascending=True)

plt.figure(figsize=(12, 7))
plt.barh(top_combo["event_type_combo"], top_combo["users"])
plt.title("Top user positive event_type combinations")
plt.xlabel("Users")
plt.ylabel("Event type combo")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_03_top_event_type_combos.png"), dpi=160)
plt.show()

# 6.4 Distribution: number of event types per user
dist_df = n_type_distribution.sort_values("n_positive_event_types")

plt.figure(figsize=(8, 5))
plt.bar(dist_df["n_positive_event_types"].astype(str), dist_df["users"])
plt.title("How many positive event types does each user use?")
plt.xlabel("Number of positive event types used by user")
plt.ylabel("Users")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_04_n_event_type_distribution.png"), dpi=160)
plt.show()

# 6.5 Heatmap overlap
plt.figure(figsize=(8, 6))
plt.imshow(overlap_matrix.values)
plt.xticks(range(len(overlap_matrix.columns)), overlap_matrix.columns, rotation=45, ha="right")
plt.yticks(range(len(overlap_matrix.index)), overlap_matrix.index)

for i in range(overlap_matrix.shape[0]):
    for j in range(overlap_matrix.shape[1]):
        plt.text(
            j,
            i,
            f"{int(overlap_matrix.iloc[i, j]):,}",
            ha="center",
            va="center"
        )

plt.title("User overlap between positive event types")
plt.colorbar(label="Users")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_05_event_type_overlap_heatmap.png"), dpi=160)
plt.show()

print("\nDONE.")
print("Output folder:", OUTPUT_DIR)
print("\nFile nên xem:")
print("01_event_type_user_summary.csv")
print("03_user_event_type_combo_summary.csv")
print("05_event_type_overlap_matrix.csv")
print("plot_01_unique_users_by_event_type.png")
print("plot_03_top_event_type_combos.png")
print("plot_05_event_type_overlap_heatmap.png")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR_OLD = "/kaggle/working/positive_interaction_stats"
OUTPUT_DIR_NEW = "/kaggle/working/positive_interaction_stats_no_other_quick"
os.makedirs(OUTPUT_DIR_NEW, exist_ok=True)

user_event_type_path = os.path.join(OUTPUT_DIR_OLD, "user_event_type_summary.parquet")

KEEP_EVENT_TYPES = [
    "view_phone",
    "contact_chat",
    "contact_zalo",
    "contact_sms",
]

uet = pd.read_parquet(user_event_type_path)
uet = uet[uet["event_type"].isin(KEEP_EVENT_TYPES)].copy()

# ============================================================
# 1. Event type summary
# ============================================================

total_positive_users = uet["user_id"].nunique()

event_type_summary = (
    uet.groupby("event_type", as_index=False)
    .agg(
        unique_users=("user_id", "nunique"),
        total_events=("event_count", "sum"),
        avg_events_per_user=("event_count", "mean"),
        median_events_per_user=("event_count", "median"),
        max_events_by_one_user=("event_count", "max"),
    )
)

event_type_summary["user_share_among_positive_users"] = (
    event_type_summary["unique_users"] / max(total_positive_users, 1)
)

event_type_summary = event_type_summary.sort_values("unique_users", ascending=False)

display(event_type_summary)

event_type_summary.to_csv(
    os.path.join(OUTPUT_DIR_NEW, "01_event_type_user_summary_no_other.csv"),
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 2. User combo summary
# ============================================================

user_combo = (
    uet.groupby("user_id")
    .agg(
        event_type_combo=("event_type", lambda x: " + ".join(sorted(set(x)))),
        n_event_types=("event_type", "nunique"),
        total_events=("event_count", "sum"),
    )
    .reset_index()
)

combo_summary = (
    user_combo.groupby(["event_type_combo", "n_event_types"], as_index=False)
    .agg(
        users=("user_id", "nunique"),
        total_events=("total_events", "sum"),
    )
)

combo_summary["user_share"] = combo_summary["users"] / combo_summary["users"].sum()
combo_summary = combo_summary.sort_values("users", ascending=False)

display(combo_summary.head(30))

combo_summary.to_csv(
    os.path.join(OUTPUT_DIR_NEW, "02_user_event_type_combo_summary_no_other.csv"),
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 3. Number of event types per user
# ============================================================

n_type_distribution = (
    user_combo.groupby("n_event_types", as_index=False)
    .agg(users=("user_id", "nunique"))
)

n_type_distribution["user_share"] = (
    n_type_distribution["users"] / n_type_distribution["users"].sum()
)

display(n_type_distribution)

n_type_distribution.to_csv(
    os.path.join(OUTPUT_DIR_NEW, "03_n_event_type_distribution_no_other.csv"),
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 4. Overlap matrix
# ============================================================

user_type_flag = (
    uet.assign(flag=1)
    .pivot_table(
        index="user_id",
        columns="event_type",
        values="flag",
        aggfunc="max",
        fill_value=0
    )
)

for et in KEEP_EVENT_TYPES:
    if et not in user_type_flag.columns:
        user_type_flag[et] = 0

user_type_flag = user_type_flag[KEEP_EVENT_TYPES].astype(int)

overlap_matrix = pd.DataFrame(
    user_type_flag.T.values @ user_type_flag.values,
    index=KEEP_EVENT_TYPES,
    columns=KEEP_EVENT_TYPES
)

display(overlap_matrix)

overlap_matrix.to_csv(
    os.path.join(OUTPUT_DIR_NEW, "04_event_type_overlap_matrix_no_other.csv"),
    encoding="utf-8-sig"
)

# ============================================================
# 5. Plots
# ============================================================

plot_df = event_type_summary.sort_values("unique_users", ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(plot_df["event_type"], plot_df["unique_users"])
plt.title("Unique users by contact event_type")
plt.xlabel("Unique users")
plt.ylabel("Event type")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR_NEW, "plot_01_unique_users_by_contact_event_type.png"), dpi=160)
plt.show()

plot_df = event_type_summary.sort_values("total_events", ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(plot_df["event_type"], plot_df["total_events"])
plt.title("Total events by contact event_type")
plt.xlabel("Total events")
plt.ylabel("Event type")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR_NEW, "plot_02_total_events_by_contact_event_type.png"), dpi=160)
plt.show()

top_combo = combo_summary.head(15).sort_values("users", ascending=True)

plt.figure(figsize=(11, 6))
plt.barh(top_combo["event_type_combo"], top_combo["users"])
plt.title("Top user contact event_type combinations")
plt.xlabel("Users")
plt.ylabel("Event type combo")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR_NEW, "plot_03_top_contact_event_type_combos.png"), dpi=160)
plt.show()

plt.figure(figsize=(7, 5))
plt.imshow(overlap_matrix.values)
plt.xticks(range(len(overlap_matrix.columns)), overlap_matrix.columns, rotation=45, ha="right")
plt.yticks(range(len(overlap_matrix.index)), overlap_matrix.index)

for i in range(overlap_matrix.shape[0]):
    for j in range(overlap_matrix.shape[1]):
        plt.text(
            j,
            i,
            f"{int(overlap_matrix.iloc[i, j]):,}",
            ha="center",
            va="center"
        )

plt.title("User overlap between contact event types")
plt.colorbar(label="Users")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR_NEW, "plot_04_contact_event_type_overlap_heatmap.png"), dpi=160)
plt.show()

print("DONE.")
print("Output folder:", OUTPUT_DIR_NEW)